<a href="https://colab.research.google.com/github/DhaiHamoud/Fraud-Detection-in-the-Financial-Transaction-Market/blob/main/notebooks/04_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import gdown, pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
gdown.download(id='11E44BQI86alA9DtSmEr3_yAiUvlzw8Sz', output='features.parquet', quiet=False)
df = pd.read_parquet('features.parquet')

Downloading...
From: https://drive.google.com/uc?id=11E44BQI86alA9DtSmEr3_yAiUvlzw8Sz
To: /content/features.parquet
100%|██████████| 47.1M/47.1M [00:00<00:00, 147MB/s]


## 1. Data Split and Baseline — Lana

In [8]:
# Time-based split: train = Jan-Sep 2019, validation = Oct-Dec 2019, test = 2020
train_part = (df['is_train'] == 1) & (df['month'] <= 9)
val_part = (df['is_train'] == 1) & (df['month'] >= 10)
test_part = df['is_train'] == 0

# Features (X) and target (y)
features = [c for c in df.columns if c not in ['is_fraud', 'is_train']]
X_train, y_train = df.loc[train_part, features], df.loc[train_part, 'is_fraud']
X_val, y_val = df.loc[val_part, features], df.loc[val_part, 'is_fraud']
X_test, y_test = df.loc[test_part, features], df.loc[test_part, 'is_fraud']

# Size and fraud rate of each part
for name, y in [('Train', y_train), ('Validation', y_val), ('Test', y_test)]:
    print(f'{name}: {len(y):,} rows, {y.sum():,} fraud ({y.mean()*100:.2f}%)')
print('Number of features:', len(features))

Train: 644,611 rows, 3,786 fraud (0.59%)
Validation: 280,239 rows, 1,434 fraud (0.51%)
Test: 927,544 rows, 4,431 fraud (0.48%)
Number of features: 25


In [9]:
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, roc_auc_score

# Calculate all metrics from the fraud probabilities (threshold 0.5 by default)
def evaluate(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred),
            'f1': f1_score(y_true, y_pred),
            'pr_auc': average_precision_score(y_true, y_prob),
            'roc_auc': roc_auc_score(y_true, y_prob)}

# Validation results of every model are stored here
results = {}

In [10]:
from sklearn.dummy import DummyClassifier

# Baseline: always predicts the most common class (legitimate)
# # Minimum performance to beat: every model that comes next should perform better than this baseline
baseline = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
results['Baseline'] = evaluate(y_val, baseline.predict_proba(X_val)[:, 1])
pd.DataFrame(results).T.round(3)

,precision,recall,f1,pr_auc,roc_auc
Baseline,0.0,0.0,0.0,0.005,0.5


Notes:

* **Data Split:**
  * **Train (Jan-Sep 2019):** 644,611 rows, 3,786 fraud cases (0.59%).
  * **Validation (Oct-Dec 2019):** 280,239 rows, 1,434 fraud cases (0.51%).
  * **Test (2020):** 927,544 rows, 4,431 fraud cases (0.48%).
  * **Features:** The data has 25 features.
  * **Fraud rate:** Fraud is rare in all three parts, and its rate falls slightly over time.

* **Time-Based Validation:**
  * **Why late 2019:** The model will be used on future transactions, so we validate it on data that comes after its training period.
  * **Why not a random split:** The model could learn from later months and then predict earlier ones, making the results look better than they would be in real use.

* **Baseline Results:**
  * **Scores:** Precision = 0.0, Recall = 0.0, F1 = 0.0, PR-AUC = 0.005, and ROC-AUC = 0.5.
  * **Why F1 is 0:** The baseline labels every transaction as legitimate, so it catches none of the 1,434 fraud cases in validation. Precision and recall are both 0, so F1 is also 0.
  * **Why PR-AUC is about the fraud rate:** The baseline gives every transaction the same score, so it can't rank fraud above legitimate transactions. Its PR-AUC therefore equals the share of fraud in the data: 0.005, or about 0.51%.

* **Minimum Performance to Beat:**
  Any useful model should score well above the baseline's PR-AUC of 0.005 and ROC-AUC of 0.5, and it should catch actual fraud cases.